# Do-as-I-Do · Reconstruction — RunPod

End-to-end hand + object **reconstruction and 6-DoF pose tracking** from a single demo video,
running the [`reconstruction/`](https://github.com/malik-group/do-as-i-do) pipeline on a RunPod
**A100 80 GB** pod, from a Jupyter notebook.

### RunPod prerequisites (do these in the RunPod dashboard / pod first)
1. **Pod template.** Launch an **NVIDIA A100 80 GB** pod with a CUDA 12.x PyTorch template
   (e.g. RunPod's official `PyTorch` template). The CUDA toolkit (`nvcc` at `/usr/local/cuda`) is
   used to compile a few extensions (pytorch3d, DROID-SLAM, nvdiffrast, ...).
2. **Network volume (recommended).** Attach a network volume at `/workspace` so the cloned repo,
   weights, video, and MANO files persist across pod restarts.
3. **Upload your inputs** onto the pod (RunPod file browser / `scp`):
   - your demo **video** (e.g. `/workspace/pipette.mp4`);
   - `MANO_RIGHT.pkl` and `MANO_LEFT.pkl` (manual, license-gated download from
     https://mano.is.tue.mpg.de) into a folder, e.g. `/workspace/mano/`.
4. **HuggingFace access.** Request access to the gated repos `facebook/sam-3d-objects` and
   `facebook/sam3`. Have a token ready (https://huggingface.co/settings/tokens) — paste it in the
   auth cell, or export it as `HF_TOKEN` before launching Jupyter.

### What this notebook does
Installs Miniconda, builds the pipeline's **4 conda envs** (with all compiled extensions baked in),
fetches weights, lets you **click the object** on the reference frame (interactive widget), and runs
`run_pipeline.sh` end-to-end with the click substituted by your points.

Run cells top-to-bottom. **[setup]** cells run once per pod; **[run]** cells are per-video.

> Tip: pick `FRAME_N` (§0) where the object is clearly visible AND the hand grips it — it drives
> both mesh reconstruction (Step 2) and hand scale-calibration (Step 4). Use the optional §13b
> frame-finder after the first run if Stage 4 complains about hand raycast hits.

## 0 · Configuration  [run]

All paths are local filesystem paths on the pod. The video is **auto-normalized** into a per-video
subdirectory (`<name>/<name>.mp4`) — the pipeline assumes `video_dir` basename == video name
(the repo convention, e.g. `whisking/whisking.mp4`), and normalizing makes every downstream path
inference line up automatically.

In [ ]:
import os

# --- Your video (path on the pod) ---
VIDEO_PATH = "/workspace/pipette.mp4"            # edit me

# --- Reference frame + object + anchor hand (same args as run_pipeline.sh) ---
FRAME_N = 42                                     # edit me
OBJECT  = "pipette"                              # edit me
ANCHOR_HAND = "right"                            # "right" or "left"

# --- MANO models (folder containing MANO_RIGHT.pkl + MANO_LEFT.pkl) ---
MANO_DIR = "/workspace/mano"                     # edit me

# --- Where to clone the repo (put it on the network volume for persistence) ---
REPO_DIR = "/workspace/do-as-i-do"

# --- Normalize the video into a per-video subdir so pipeline path inferences align ---
VIDEO_PATH = os.path.abspath(VIDEO_PATH)
vstem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
vext  = os.path.splitext(VIDEO_PATH)[1]
parent = os.path.dirname(VIDEO_PATH)
if os.path.basename(parent) != vstem:
    sub = os.path.join(parent, vstem)
    os.makedirs(sub, exist_ok=True)
    link = os.path.join(sub, vstem + vext)
    if not os.path.exists(link):
        os.symlink(VIDEO_PATH, link)
    VIDEO_PATH = link
    print(f"normalized video -> {VIDEO_PATH}")

for k, v in {"VIDEO_PATH": VIDEO_PATH, "FRAME_N": str(FRAME_N), "OBJECT": OBJECT,
             "ANCHOR_HAND": ANCHOR_HAND, "MANO_DIR": MANO_DIR, "REPO_DIR": REPO_DIR}.items():
    os.environ[k] = v
print("VIDEO_PATH =", VIDEO_PATH)
print("REPO_DIR   =", REPO_DIR)

## 1 · GPU, disk & system dependencies  [setup]

Checks the GPU (≥ 32 GB VRAM, `nvcc` present) and installs the **system packages** several stages
need once: `rsync` (weight fetch), `wget`, `zip` (download), and the **EGL/OpenGL libs** HaWoR's
headless renderer requires (`libegl1 libgl1 libgles2`). Doing this up front avoids mid-run failures.

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
VRAM_MB=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1 | tr -d ' ')
[ "${VRAM_MB:-0}" -lt 30000 ] && echo "!! needs >= 32 GB VRAM (have ${VRAM_MB} MB)"
echo "--- nvcc ---"
nvcc --version 2>/dev/null || ls /usr/local/cuda/bin/nvcc 2>/dev/null || echo "nvcc MISSING — use a CUDA dev image"
echo "--- system deps ---"
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq && apt-get install -y -qq rsync wget zip libegl1 libgl1 libgles2
df -h /workspace 2>/dev/null || df -h /

## 2 · Notebook-kernel deps + HuggingFace login  [setup]

Kernel-side packages for the interactive click widget (§11). Then authenticate to HuggingFace so
the gated `facebook/sam-3d-objects` / `facebook/sam3` checkpoints download.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ipyevents", "ipywidgets", "ipympl", "matplotlib", "opencv-python", "numpy"], check=True)
print("kernel deps installed.")

In [ ]:
import os, getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your HuggingFace token (input hidden): ")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("HF token set (length %d)." % len(os.environ["HF_TOKEN"]))

In [ ]:
%%bash
pip install -q 'huggingface-hub[cli]<1.0'
git config --global credential.helper store
hf auth login --token "$HF_TOKEN" --add-to-git-credential

## 3 · Install Miniconda  [setup]

No-op if the pod image already has conda at `/opt/conda`; otherwise installs Miniconda. Every later
`%%bash` cell re-sources it (shell state does not persist between cells).

In [ ]:
%%bash
set -e
if [ -x /opt/conda/bin/conda ]; then echo "conda already at /opt/conda"; else
  cd /tmp && wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O m.sh
  bash m.sh -bfp /opt/conda && rm m.sh
fi
source /opt/conda/etc/profile.d/conda.sh
conda --version
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true

## 4 · Clone the repo + submodules  [setup]

`GIT_LFS_SKIP_SMUDGE=1` so weight blobs are not pulled by Git LFS — they come from the weight-fetch
step later.

In [ ]:
%%bash
set -e
if [ -d "$REPO_DIR/.git" ]; then echo "repo already at $REPO_DIR"; else
  mkdir -p "$(dirname "$REPO_DIR")"; cd "$(dirname "$REPO_DIR")"
  GIT_LFS_SKIP_SMUDGE=1 git clone --recurse-submodules https://github.com/malik-group/do-as-i-do.git "$(basename "$REPO_DIR")"
fi
cd "$REPO_DIR"
GIT_LFS_SKIP_SMUDGE=1 git submodule update --init --recursive
git submodule status

## 5 · Build the 4 conda envs  [setup]

Each env bakes in **everything** it needs (compiled extensions included), so the run phase needs no
extra installs:

- **`sam3`** / **`tapnet`** — `setup/01_create_envs.sh` (cu128 torch, fine on A100).
- **`sam3d`** — `01_create_envs.sh` + the `+cu121` wheel indices, plus the compiled extensions the
  sam-3d-objects `[p3d]`/`[inference]` extras provide (`flash-attn`, `nvdiffrast`, `pytorch3d`,
  Mip-Splatting `diff_gaussian_rasterization`), the vendored-`notebook` un-shadow, `geocalib`,
  and `viser`. Built `--no-build-isolation` against the env's torch for sm_80.
- **`hawor`** — torch 1.13+cu117 with a **matching CUDA 11.7 toolkit** in-env (torch 1.13 enforces
  an exact nvcc match), `--no-build-isolation` builds (pytorch3d, torch-scatter import torch),
  `setuptools<81` (pkg_resources), PyG wheel index for torch-scatter, and the DROID-SLAM gencode
  patched from the fork's hardcoded `compute_120` to A100 `compute_80`.

These are slow (~5-30 min each). Re-run a failed cell — pip/conda resume satisfied packages.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
bash setup/01_create_envs.sh sam3

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
SAM3D_DIR="$REPO_DIR/reconstruction/modules/sam-3d-objects"

# sam-3d-objects/requirements.txt pins torch 2.5.1/torchvision/torchaudio with +cu121, and kaolin
# comes from NVIDIA's wheel bucket. setup/01_create_envs.sh doesn't set these (mirrors doc/setup.md).
export PIP_EXTRA_INDEX_URL="https://pypi.ngc.nvidia.com https://download.pytorch.org/whl/cu121"
export PIP_FIND_LINKS="https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.5.1_cu121.html"
export CUDA_HOME="${CUDA_HOME:-/usr/local/cuda}"
export TORCH_CUDA_ARCH_LIST="8.0"     # A100 (sm_80)
export FORCE_CUDA=1
bash setup/01_create_envs.sh sam3d

conda activate sam3d
pip install "setuptools<81" ninja wheel packaging einops psutil

# Un-shadow sam-3d-objects' vendored notebook/ (namespace pkg under $SAM3D_DIR) from the PyPI
# `notebook` (a regular package that wins per PEP 420).
pip uninstall -y notebook notebook_shim jupyterlab jupyter_server 2>/dev/null || true

# Compiled extensions from the [p3d]/[inference] extras (all import torch in setup.py ->
# --no-build-isolation).
pip install flash-attn --no-build-isolation
pip install --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git
pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git@stable"

# Mip-Splatting diff_gaussian_rasterization (the 'inria' GLB/texture baking backend).
python - <<'PY'
import importlib.util as u, subprocess, os
if u.find_spec("diff_gaussian_rasterization") is None:
    work = "/workspace/mip-splatting-build"
    if not os.path.isdir(work):
        subprocess.run(["git", "clone", "--recursive",
                        "https://github.com/autonomousvision/mip-splatting.git", work], check=True)
    sub = os.path.join(work, "submodules", "diff-gaussian-rasterization")
    e = os.environ.copy(); e["CUDA_HOME"] = os.environ.get("CUDA_HOME") or "/usr/local/cuda"
    e["FORCE_CUDA"] = "1"; e["TORCH_CUDA_ARCH_LIST"] = "8.0"
    subprocess.run(["python", "setup.py", "install"], cwd=sub, env=e, check=True)
    print("diff_gaussian_rasterization built.")
else:
    print("diff_gaussian_rasterization already present.")
PY

pip install "geocalib @ git+https://github.com/cvg/GeoCalib.git"
pip install viser

# Verify (notebook.inference must be imported from $SAM3D_DIR — the CWD generate_mesh_sam3d.py runs from).
( cd "$SAM3D_DIR" && python -c "import notebook.inference; print('notebook.inference OK')" )
python -c "import pytorch3d, flash_attn, nvdiffrast; print('sam3d compiled exts OK')"

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
RECON="$REPO_DIR/reconstruction"
HAWOR_DIR="$RECON/modules/HaWoR"
cd "$RECON"

# torch 1.13 (cu117) does an EXACT nvcc-vs-torch CUDA check before compiling extensions; the pod's
# system nvcc is 12.x -> mismatch. Install a matching CUDA 11.7 toolkit into the env and build with it.
if ! conda env list | grep -q '^hawor '; then conda create -y -n hawor python=3.10; fi
conda activate hawor
conda install -y -c conda-forge ffmpeg
conda install -y -c "nvidia/label/cuda-11.7.1" cuda-toolkit

pip install torch==1.13.0+cu117 torchvision==0.14.0+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
# setuptools<81 BEFORE building: torch 1.13 cpp_extension does `from pkg_resources import packaging`,
# removed in setuptools>=81.
pip install "setuptools<81" ninja wheel

# requirements.txt with --no-build-isolation: pytorch3d / torch-scatter import torch in setup.py.
# PyG's wheel index lets torch-scatter use a prebuilt wheel; pytorch3d still compiles for sm_80.
export CUDA_HOME="$CONDA_PREFIX"
export PATH="$CUDA_HOME/bin:$PATH"
export TORCH_CUDA_ARCH_LIST="8.0"
export FORCE_CUDA=1
export PIP_FIND_LINKS="https://data.pyg.org/whl/torch-1.13.0+cu117.html"
grep -viE "mmcv==1.3.9|chumpy@" "$HAWOR_DIR/requirements.txt" | pip install -r /dev/stdin --no-build-isolation

pip install "chumpy@git+https://github.com/mattloper/chumpy" --no-build-isolation
pip install pytorch-lightning==2.2.4 --no-deps
pip install lightning-utilities torchmetrics==1.4.0

# DROID-SLAM: the fork's setup.py hardcodes compute_120 (Blackwell) gencode, ignored by our env var
# and rejected by nvcc 11.7. Patch to A100 sm_80 before building.
sed -i 's/compute_120,code=sm_120/compute_80,code=sm_80/g; s/compute_120,code=compute_120/compute_80,code=compute_80/g' \
  "$HAWOR_DIR/thirdparty/DROID-SLAM/setup.py"
( cd "$HAWOR_DIR/thirdparty/DROID-SLAM" && python setup.py install )

# torch>=2.6 defaults weights_only=True, which rejects HaWoR's omegaconf-bearing ckpts.
conda env config vars set TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 -n hawor
python -c "import torch,pytorch3d; print('hawor: torch', torch.__version__)"

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
bash setup/01_create_envs.sh tapnet

## 6 · Fetch model weights  [setup]

Runs `setup/02_fetch_weights.sh --download` (SAM3D from HuggingFace, HaWoR/Metric3D/DROID, BootsTAPIR).
System deps (`rsync`, `wget`) were installed in §1. The Stage-1 SAM3 model auto-downloads at runtime.

In [ ]:
%%bash
set -eo pipefail
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
bash setup/02_fetch_weights.sh --download

## 7 · Place MANO hand models  [setup]

MANO is license-gated; copy the two `.pkl` files (uploaded to `MANO_DIR`) into HaWoR's expected paths.

In [ ]:
import os, shutil, sys
HAWOR = os.path.join(os.environ["REPO_DIR"], "reconstruction/modules/HaWoR")
targets = {"MANO_RIGHT.pkl": f"{HAWOR}/_DATA/data/mano/MANO_RIGHT.pkl",
           "MANO_LEFT.pkl":  f"{HAWOR}/_DATA/data_left/mano_left/MANO_LEFT.pkl"}
missing = []
for name, dst in targets.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    src = os.path.join(os.environ["MANO_DIR"], name)
    if os.path.isfile(src): shutil.copy2(src, dst); print(f"placed {name}")
    elif os.path.isfile(dst): print(f"{name} already present")
    else: missing.append(src)
if missing:
    print("!! missing:", missing); sys.exit(1)
print("MANO in place.")

## 8 · Setup sanity check  [setup]

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh
conda env list
echo "=== torch per env ==="
for e in sam3 sam3d hawor tapnet; do
  echo -n "$e: "; conda run -n "$e" python -c "import torch;print(torch.__version__,'cuda='+str(torch.version.cuda))" 2>/dev/null || echo "(failed)"
done
R="$REPO_DIR/reconstruction"
echo "=== weights ==="
ls -lh "$R/weights/tapnet/bootstapir_checkpoint_v2.pt" 2>/dev/null || echo "MISSING tapnet ckpt"
ls -lh "$R/modules/HaWoR/weights/hawor/checkpoints/hawor.ckpt" 2>/dev/null || echo "MISSING hawor.ckpt"
ls -lh "$R/modules/HaWoR/_DATA/data/mano/MANO_RIGHT.pkl" 2>/dev/null || echo "MISSING MANO_RIGHT.pkl"
echo "=== sam3d compiled exts ==="
conda run -n sam3d python -c "import pytorch3d,flash_attn,nvdiffrast; print('OK')" 2>/dev/null || echo "(sam3d exts missing)"

---

# Run phase (per-video)

Re-run from here whenever you change `VIDEO_PATH` / `FRAME_N` / etc.

## 9 · Extract the reference frame for clicking  [run]

Extracts all frames (Step 0 of `run_pipeline.sh`) and the single reference frame. `-nostdin` +
`</dev/null` is **critical** in a `%%bash` cell — ffmpeg otherwise reads the cell's stdin and eats
the rest of the script. `-fps_mode passthrough` is the non-deprecated equivalent of `-vsync 0`.

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh
conda activate sam3
VIDEO_DIR="$(dirname "$VIDEO_PATH")"
mkdir -p "$VIDEO_DIR/all_frames"
ffmpeg -y -nostdin -i "$VIDEO_PATH" -fps_mode passthrough -start_number 0 "$VIDEO_DIR/all_frames/%06d.png" </dev/null
REF="$VIDEO_DIR/$(printf '%04d.png' "$FRAME_N")"
cp "$VIDEO_DIR/all_frames/$(printf '%06d.png' "$FRAME_N")" "$REF"
echo "reference frame: $REF"; ls -la "$REF"

## 10 · Click the object on the reference frame  [run]

Click 1-3 points on the object (green dots). Non-blocking, event-driven (`ipyevents`), so it works
in RunPod's Jupyter where blocking `ginput`/ipympl does not. If the widget doesn't appear the first
time, do one page reload (ipyevents registers its frontend on load; kernel state is preserved).

In [ ]:
import subprocess, sys, os, cv2
import ipywidgets as widgets
from ipyevents import Event
from IPython.display import display

FRAME_N = int(os.environ["FRAME_N"])
VIDEO_DIR = os.path.dirname(os.environ["VIDEO_PATH"])
REF_PNG = os.path.join(VIDEO_DIR, f"{FRAME_N:04d}.png")
img = cv2.cvtColor(cv2.imread(REF_PNG), cv2.COLOR_BGR2RGB)
assert img is not None, f"could not read {REF_PNG}"
H, W = img.shape[:2]
DISPLAY_W = min(W, 900); SCALE = W / DISPLAY_W

clicks = []
w_img = widgets.Image(value=cv2.imencode(".png", img)[1].tobytes(), format="png")
w_img.layout.width = f"{DISPLAY_W}px"
def _render():
    c = img.copy()
    for (x,y) in clicks:
        cv2.circle(c,(x,y),8,(0,255,0),-1); cv2.circle(c,(x,y),8,(255,255,255),1)
    w_img.value = cv2.imencode(".png", c)[1].tobytes()
out = widgets.Output()
def _on(event):
    nx = max(0,min(W-1,round(event["relativeX"]*SCALE))); ny = max(0,min(H-1,round(event["relativeY"]*SCALE)))
    clicks.append((nx,ny)); _render()
    with out: print(f"point {len(clicks)}: ({nx},{ny})")
click_listener = Event(source=w_img, watched_events=["click"]); click_listener.on_dom_event(_on)
done = widgets.Button(description="Done — run next cell")
done.on_click(lambda _: (out.append_stdout(f"=== {len(clicks)} points: {clicks} ===\n")))
display(widgets.VBox([w_img,
    widgets.HTML(f"<b>Click 1-3 points on '{os.environ['OBJECT']}'</b> ({W}x{H}, shown {DISPLAY_W}px)"),
    done, out]))

In [ ]:
import os
assert clicks, "click at least one point on the object above"
OBJ_POINTS       = ";".join(f"{x},{y}" for x,y in clicks)
OBJ_POINT_LABELS = ";".join("1" for _ in clicks)
print("Object points :", OBJ_POINTS)
print("Labels        :", OBJ_POINT_LABELS)
os.environ["OBJ_POINTS"] = OBJ_POINTS
os.environ["OBJ_POINT_LABELS"] = OBJ_POINT_LABELS

## 11 · Run the full pipeline  [run]

Generates `run_pipeline_colab.sh` from the repo's `run_pipeline.sh` with two string patches (the
tracked script itself is left untouched, so `git pull` never conflicts), then runs the full driver:

1. Swap the interactive `--click` for `--points` (your clicks).
2. Pass `--hand-meshes` explicitly to the Step-4 optimizer — its `--video-dir` inference guesses the
   HaWoR folder from the parent-directory name, which breaks if the video isn't in a `<name>/` subdir.

`MPLBACKEND=Agg` prevents the §10 `%matplotlib widget` backend from leaking into the bash cell and
crashing HaWoR's older matplotlib. EGL libs were installed in §1.

In [ ]:
# Patch a copy of run_pipeline.sh into run_pipeline_colab.sh. The tracked repo scripts are left
# UNTOUCHED (so `git pull` never conflicts); every notebook-specific fixup lives here as a string edit.
#
#   Fix 1: swap the single `--click` token for a --points/--point_labels pair, so Stage 1 runs
#          headlessly using the clicks you collected above.
#   Fix 2: pass --hand-meshes explicitly to the Step-4 optimizer. optimize_translation_scale.py
#          infers the HaWoR folder as basename(VIDEO_DIR) (the parent-dir name, not the video
#          stem). §0 normalizes the video into <name>/<name>.mp4 so those usually match, but if the
#          video ends up directly under VIDEO_DIR (e.g. /workspace) it looks for the nonexistent
#          <VIDEO_DIR>/<parent>/all_hand_meshes.npz and aborts the last stage. run_pipeline.sh's own
#          $HAND_MESHES_PATH uses $VIDEO_NAME (the video stem) correctly, so we hand that to the
#          optimizer — with a `find` fallback for safety (same approach as §13a).
import os
recon = os.path.join(os.environ["REPO_DIR"], "reconstruction")
src = open(f"{recon}/run_pipeline.sh").read()

# --- Fix 1: headless clicks --------------------------------------------------------------------
assert src.count("--click") == 1, f"expected exactly one --click in run_pipeline.sh, found {src.count('--click')}"
src = src.replace("--click", '--points "$OBJ_POINTS" --point_labels "$OBJ_POINT_LABELS"')

# --- Fix 2: explicit hand-meshes path for the Step-4 optimizer ---------------------------------
opt_echo = 'echo "=== Optimizing translation/scale for $OBJ_NAME ==="'
assert src.count(opt_echo) == 1, "expected exactly one Step-4 optimize echo in run_pipeline.sh"
src = src.replace(
    opt_echo,
    opt_echo + "\n"
    '    HAND_MESHES_RESOLVED="$HAND_MESHES_PATH"\n'
    '    [ -f "$HAND_MESHES_RESOLVED" ] || '
    'HAND_MESHES_RESOLVED="$(find "$VIDEO_DIR" -name all_hand_meshes.npz -print -quit)"',
)
assert src.count('--ref-frame "$n"') == 1, "expected exactly one optimizer --ref-frame in run_pipeline.sh"
src = src.replace(
    '--ref-frame "$n"',
    '--ref-frame "$n" \\\n        --hand-meshes "$HAND_MESHES_RESOLVED"',
)

open(f"{recon}/run_pipeline_colab.sh", "w").write(src)
print("wrote run_pipeline_colab.sh (patched: --points, --hand-meshes)")

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
chmod +x run_pipeline_colab.sh
export OBJ_POINTS="$OBJ_POINTS"
export OBJ_POINT_LABELS="$OBJ_POINT_LABELS"
export MPLBACKEND=Agg
echo "=== launching pipeline ==="
./run_pipeline_colab.sh "$VIDEO_PATH" "$FRAME_N" "$OBJECT" "$ANCHOR_HAND"

## 12 · Done — where the outputs live  [run]

Outputs are written next to the video under `$VIDEO_DIR/` (the per-video subdir from §0). The
retargeting input is:
```
<VIDEO_DIR>/obj_tracking_out/<OBJECT>/combined_visualization/layout_camera_frame_optimized.json
```

In [ ]:
%%bash
VIDEO_DIR="$(dirname "$VIDEO_PATH")"
find "$VIDEO_DIR" -maxdepth 3 -type d | sort
echo; ls -la "$VIDEO_DIR/obj_tracking_out/$OBJECT/combined_visualization/layout_camera_frame_optimized.json" 2>/dev/null \
  && echo "OK: reconstruction finished" \
  || echo "!! optimized layout missing — see log / use §13a to resume."

---

## 13 · Recovery utilities (optional)

Use these only if the main run (§11) fails partway and you want to avoid redoing the slow stages:
- **§13a** resumes from the Step-3 tail (skips the slow `track_object`).
- **§13b** finds the reference frame with the best hand↔mask alignment (pick it when Stage 4 reports
  "too few hand raycast hits" — that means HaWoR's pose was poor at your `FRAME_N`).

### 13a · Resume from the Step-3 tail (project / convert / optimize)  [run]

`track_object` already finished (its outputs are in `obj_tracking_out/`). This runs only the fast
Step-3 tail + Step 4, and decouples the object mesh (reconstructed at your original `FRAME_N`) from
the calibration frame, so you can change `FRAME_N` for calibration without redoing mesh recon.

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
cd "$REPO_DIR/reconstruction"
source config/paths.sh

VIDEO_PATH="$(realpath "$VIDEO_PATH")"
n="$FRAME_N"; OBJECT_ID="${OBJECT// /_}"; ANCHOR_HAND="$ANCHOR_HAND"
VIDEO_DIR="$(dirname "$VIDEO_PATH")"; VIDEO_NAME="$(basename "${VIDEO_PATH%.*}")"
CV="$VIDEO_DIR/obj_tracking_out/$OBJECT_ID/combined_visualization"
export MPLBACKEND=Agg

# Object mesh may be at a different frame than the current calibration FRAME_N — locate it.
OBJ_MESH="$(find "$VIDEO_DIR/video_segmentation/masks" -name "${OBJECT_ID}.obj" -print -quit)"
HAND_MESHES="$VIDEO_DIR/$VIDEO_NAME/all_hand_meshes.npz"
[ -f "$(realpath "$HAND_MESHES" 2>/dev/null || echo /x)" ] || HAND_MESHES="$(find "$VIDEO_DIR" -name all_hand_meshes.npz -print -quit)"
echo "mesh=$OBJ_MESH"; echo "hand_meshes=$HAND_MESHES"; test -f "$OBJ_MESH"; test -f "$HAND_MESHES"

conda activate "$ENV_SAM3D"; cd "$SCRIPTS_DIR"
[ -f "$CV/layout_camera_frame.json" ] || {
  python run_project_mesh_combined.py --video "$VIDEO_PATH" --mesh "$OBJ_MESH" \
      --json "$CV/layout.json" --output-base "$CV/projected"
  python convert_layout_to_camera_frame.py --input "$CV/layout.json" --output "$CV/layout_camera_frame.json"
}
python optimize_translation_scale.py --video-dir "$VIDEO_DIR" \
    --layout-json "$CV/layout_camera_frame.json" --mesh "$OBJ_MESH" --hand-meshes "$HAND_MESHES" \
    --anchor-hand "$ANCHOR_HAND" --ref-frame "$n"
ls -la "$CV/layout_camera_frame_optimized.json"

### 13b · Find the best reference frame  [run]

Ranks every frame by how well the HaWoR hand mesh projects into the hand mask. Pick the top frame
as `FRAME_N` (§0) and re-run — Stage 4 needs the hand to align at the reference frame.

In [ ]:
import numpy as np, cv2, os
VD = os.path.dirname(os.environ["VIDEO_PATH"])
VNAME = os.path.basename(os.path.splitext(os.environ["VIDEO_PATH"])[0])
ANCHOR = os.environ["ANCHOR_HAND"]; OBJECT = os.environ["OBJECT"]
hm = np.load(f"{VD}/{VNAME}/all_hand_meshes.npz")
V = hm[f"{ANCHOR}_vertices"]
# use the reference-frame pointmap intrinsics if present, else fall back to image-center guess
import glob
def intr(fidx):
    for p in [f"{VD}/{fidx:04d}_intrinsics.txt"]+glob.glob(f"{VD}/all_frames/{fidx:06d}_intrinsics*"):
        if os.path.exists(p):
            if p.endswith('.txt'):
                fx,fy,cx,cy=[float(x) for x in open(p).read().split()[:4]]; return fx,fy,cx,cy
            K=np.load(p); K=K.reshape(3,3); return K[0,0],K[1,1],K[0,2],K[1,2]
    return None
rows=[]
for fi in range(len(V)):
    K=intr(fi)
    if K is None: continue
    fx,fy,cx,cy=K
    mp=f"{VD}/video_segmentation/masks/frame_{fi:06d}_masks/{ANCHOR}_hand_0.png"
    if not os.path.exists(mp): continue
    m=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
    if (m>0).sum()<500: continue
    v=V[fi]; z=v[:,2]; pos=z>0
    if not pos.any(): continue
    u=(fx*v[pos,0]/v[pos,2]+cx).astype(int); w=(fy*v[pos,1]/v[pos,2]+cy).astype(int)
    ok=(u>=0)&(u<m.shape[1])&(w>=0)&(w<m.shape[0])
    inn=np.zeros(int(pos.sum()),bool); inn[ok]=m[w[ok],u[ok]]>0
    rows.append((fi,int((m>0).sum()),int(inn.sum())))
rows.sort(key=lambda r:-r[2])
print("best frames (fi, mask_px, mesh_verts_in_mask):")
for r in rows[:12]: print(" ",r)

## 14 · Interactive 3D visualization (viser)  [run]

Launches the viser web viewer (object mesh + tracked pose + both hands) on `:8080`; it **blocks**.
Reach it from your laptop via SSH port forwarding (run on your **local** machine):
```
ssh -N -L 8080:localhost:8080 root@<pod-ip> -p <port> -i <key>
```
then open `http://localhost:8080`. Stop the cell (■) when done.

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh
conda activate sam3d
cd "$REPO_DIR/reconstruction"
VIDEO_DIR="$(dirname "$VIDEO_PATH")"; n="$FRAME_N"; OBJECT_ID="${OBJECT// /_}"
VIDEO_NAME="$(basename "${VIDEO_PATH%.*}")"
CV="/workspace/obj_tracking_out/$OBJECT_ID/combined_visualization"
HAND_MESHES="/workspace/$VIDEO_NAME/all_hand_meshes.npz"
OBJ_MESH="$(find "/workspace/video_segmentation/masks" -name "${OBJECT_ID}.obj" -print -quit)"
MESH_SCALE="$(python3 -c "import json;d=json.load(open('$CV/layout_camera_frame_optimized.json'));print(d['translation_scale_optimization']['mesh_scale'])")"
echo "mesh=$OBJ_MESH scale=$MESH_SCALE"
python scripts/visualize_3d.py \
    --frames-dir "$VIDEO_DIR/all_frames" --layout-json "$CV/layout_camera_frame_optimized.json" \
    --mesh "$OBJ_MESH" --scale "$MESH_SCALE" --translation-scale 1.0 \
    --hand-meshes "$HAND_MESHES" --port 8080 </dev/null

## 15 · Zip & download results  [run]

Zips everything under the video's directory (excluding the heavy, regenerable `do-as-i-do/` repo+
weights and `mip-splatting-build/`). Download with `scp -P <port> -i <key> root@<pod>:/workspace/<name>.zip .`

In [ ]:
%%bash
set -e
VNAME="$(basename "${VIDEO_PATH%.*}")"
OUT="/workspace/${VNAME}_results.zip"
cd "/workspace/" && zip -r -q "$OUT" "$VNAME" \
  -x "*/do-as-i-do/*" "*/mip-splatting-build/*" "*/.ipynb_checkpoints/*"
ls -lh "$OUT"